## Annotate the new data's latent scores with pathways and drugs

Links each new sample's BioBombe latent scores to the same GSEA pathway
results and drug correlations used everywhere else in this project, via
`utils/pinwheels.py`'s `compute_and_plot_latent_scores` /
`assign_unique_latent_dims` — the same functions
`6.collab-data/scripts/3.pinwheel_plots.py` uses for collaborator data.

Run this script with `8.apply-biobombe-new-data/` as the working directory.

In [1]:
import pathlib
import sys

import pandas as pd

sys.path.insert(0, "../utils/")
from pinwheels import assign_unique_latent_dims, compute_and_plot_latent_scores

In [2]:
data_results_dir = pathlib.Path("data/largaespada/results")

combined_latent_df = pd.read_parquet(data_results_dir / "NF1_data_latent_scores.parquet")
print(f"Latent scores: {combined_latent_df.shape}")

Latent scores: (3888, 408)


### Pathway annotation: CORUM and Reactome, kept separate

A real, ensemble-wide Reactome run did happen for this project at some
point — 4.drug-dependency/results/all_reactome_results.parquet holds genuine
Reactome pathways (R-HSA-tagged names, e.g. "rRNA Modification In Nucleus
And Cytosol R-HSA-6790901") spanning all 6 model types, so this isn't
hypothetical. What's missing is the raw, per-latent-dimension GSEA hit table
that would have fed it (the direct analog of combined_z_matrix_gsea_results.parquet,
but for Reactome) — it isn't in 3.run-biobombe/gsea_results/ or anywhere
else in this repo's git history. `reactome_gsea_path` below is where
`2.interpret_biobombe_with_gsea.py` would need to write one (by re-running
it with a Reactome library) for that half to populate again.

`3.run-biobombe/gsea_results/combined_z_matrix_gsea_results.parquet` — the
one raw file that does exist — is CORUM, confirmed directly: every one of
its 170 pathway names matches CORUM's complex-naming convention ("X complex
(human)"), never Reactome's R-HSA- IDs. `4.drug-dependency/scripts/7.pinwheel_plots.py`
actually has this backwards (it labels this file `reactome_df` and expects
a nonexistent "_corum"-suffixed file for CORUM) — that script has never
successfully run against its current code, since the file it wants for
CORUM was never committed either.

Both results, when present, use a column literally called "reactome_pathway"
for the pathway name — that's `2.interpret_biobombe_with_gsea.py`'s schema
regardless of which library was actually run, not a labeling mistake here.

### Pathway annotation (CORUM GSEA results)

In [3]:
gsea_sources = {
    "CORUM": pathlib.Path("../3.run-biobombe/gsea_results/combined_z_matrix_gsea_results.parquet"),
    "Reactome": pathlib.Path("../3.run-biobombe/gsea_results/combined_z_matrix_gsea_results_reactome.parquet"),
}

pathway_max_by_library = {}
for library, gsea_path in gsea_sources.items():
    if not gsea_path.exists():
        print(f"{library}: {gsea_path} not found, skipping.")
        continue

    gsea_df = pd.read_parquet(gsea_path)
    # Drop the shuffled negative-control rows and keep only nominally significant
    # hits. This file has no fdr column (unlike the single-VAE GSEA results
    # elsewhere in the project), so significance here is by p_value alone.
    significant_gsea_df = gsea_df[(~gsea_df["shuffled"]) & (gsea_df["p_value"] < 0.05)].copy()
    print(f"{library}: {significant_gsea_df.shape[0]} / {gsea_df.shape[0]} significant hits")

    # nes_score (normalized enrichment score), not gsea_es_score: NES adjusts
    # for gene-set-size effects, and it's what the previously-run ensemble-wide
    # annotation (4.drug-dependency/results/all_corum_results.parquet /
    # all_reactome_results.parquet) actually used.
    pathway_max_by_library[library] = assign_unique_latent_dims(
        significant_gsea_df, score_col="nes_score", target_col="reactome_pathway"
    )

CORUM: 52386 / 1142910 significant hits
Processing pathway: C complex spliceosome (human)
Assigned latent dimension 2 for model vanillavae, 18, init 2 to pathway C complex spliceosome (human).
Processing pathway: Mediator complex (human)
Assigned latent dimension 2 for model betavae, 10, init 1 to pathway Mediator complex (human).
Processing pathway: Spliceosome (human)
Assigned latent dimension 2 for model betavae, 3, init 0 to pathway Spliceosome (human).
Processing pathway: DAB complex (human)
Assigned latent dimension 5 for model betatcvae, 12, init 1 to pathway DAB complex (human).
Processing pathway: DA complex (human)
Assigned latent dimension 45 for model nmf, 200, init 0 to pathway DA complex (human).
Processing pathway: BRM-SIN3A-HDAC complex (human)


Assigned latent dimension 2 for model betatcvae, 2, init 0 to pathway BRM-SIN3A-HDAC complex (human).
Processing pathway: STAGA complex (SPT3-TAF9-GCN5 acetyltransferase complex) (human)
Assigned latent dimension 40 for model betatcvae, 50, init 1 to pathway STAGA complex (SPT3-TAF9-GCN5 acetyltransferase complex) (human).
Processing pathway: Polycystin-1 multiprotein complex (ACTN1, CDH1, SRC, JUP, VCL, CTNNB1, PXN, BCAR1, PKD1, PTK2, TLN1) (human)
Assigned latent dimension 35 for model betatcvae, 80, init 4 to pathway Polycystin-1 multiprotein complex (ACTN1, CDH1, SRC, JUP, VCL, CTNNB1, PXN, BCAR1, PKD1, PTK2, TLN1) (human).
Processing pathway: mRNA decay complex (UPF1, UPF2, UPF3B, DCP2, XRN1, XRN2, EXOSC2, EXOSC4, EXOSC10, PARN) (human)
Assigned latent dimension 5 for model betatcvae, 16, init 4 to pathway mRNA decay complex (UPF1, UPF2, UPF3B, DCP2, XRN1, XRN2, EXOSC2, EXOSC4, EXOSC10, PARN) (human).
Processing pathway: BRG1-SIN3A complex (human)
Assigned latent dimension 4 for m

Assigned latent dimension 8 for model betatcvae, 70, init 1 to pathway Emerin complex 24 (human).
Processing pathway: BRAFT complex (human)
Assigned latent dimension 16 for model vanillavae, 200, init 0 to pathway BRAFT complex (human).
Processing pathway: NuA4/Tip60-HAT complex B (human)
Assigned latent dimension 3 for model betatcvae, 35, init 1 to pathway NuA4/Tip60-HAT complex B (human).
Processing pathway: Nup 107-160 subcomplex (human)
Assigned latent dimension 4 for model betatcvae, 9, init 4 to pathway Nup 107-160 subcomplex (human).
Processing pathway: DRIP complex (human)
Assigned latent dimension 16 for model betatcvae, 200, init 4 to pathway DRIP complex (human).
Processing pathway: BAF complex (human)
Assigned latent dimension 100 for model betatcvae, 150, init 3 to pathway BAF complex (human).
Processing pathway: ARC-L complex (human)
Assigned latent dimension 3 for model betatcvae, 10, init 2 to pathway ARC-L complex (human).
Processing pathway: cMYC-ATPase-helicase comp

Assigned latent dimension 5 for model nmf, 9, init 0 to pathway Respiratory chain complex I (early intermediate NDUFAF1 assembly), mitochondrial (human).
Processing pathway: Kaiso-NCOR complex (human)
Assigned latent dimension 6 for model betatcvae, 25, init 0 to pathway Kaiso-NCOR complex (human).
Processing pathway: Kinase maturation complex 1 (human)
Assigned latent dimension 50 for model pca, 60, init 0 to pathway Kinase maturation complex 1 (human).
Processing pathway: RNA polymerase II holoenzyme complex (human)
Assigned latent dimension 3 for model betatcvae, 18, init 1 to pathway RNA polymerase II holoenzyme complex (human).
Processing pathway: Large Drosha complex (human)
Assigned latent dimension 25 for model betatcvae, 60, init 1 to pathway Large Drosha complex (human).
Processing pathway: PYR complex (mouse)
Assigned latent dimension 90 for model betavae, 200, init 3 to pathway PYR complex (mouse).
Processing pathway: SNF2h-cohesin-NuRD complex (human)
Assigned latent dimen

In [4]:
pathway_results = []
for library, pathway_max in pathway_max_by_library.items():
    for model_id in combined_latent_df["ModelID"].unique():
        df = compute_and_plot_latent_scores(
            model_id, combined_latent_df, pathway_max, "reactome_pathway", "nes_score",
            library, make_plot=False,
        )
        df["gene_set_library"] = library
        pathway_results.append(df)

pathway_results_df = pd.concat(pathway_results, ignore_index=True) if pathway_results else pd.DataFrame()
print(f"Pathway annotations (CORUM + Reactome combined): {pathway_results_df.shape}")

S462


ST88-14
STS-26T
iHSC1L_C3
iHSC1L_F12


iPN1L
iPN2J
iPN3D
Pathway annotations (CORUM + Reactome combined): (168, 11)


### Drug annotation

This step needs `4.drug-dependency/results/combined_latent_drug_correlations.parquet`,
the ensemble-wide drug correlation table `4.drug-dependency/scripts/5.drug_latent_dim_correlation.py`
produces. That script depends on `saved_models/`, which only exists locally
now — if it hasn't been run yet, drug annotation is skipped here rather than failing.

In [5]:
drug_correlation_path = pathlib.Path("../4.drug-dependency/results/combined_latent_drug_correlations.parquet")

if drug_correlation_path.exists():
    drug_df = pd.read_parquet(drug_correlation_path)
    significant_drug_df = drug_df[
        (drug_df["pearson_correlation"].abs() > 0.15) & (drug_df["p_value"] < 0.05)
    ].copy()
    drug_max = assign_unique_latent_dims(
        significant_drug_df, score_col="pearson_correlation", target_col="name"
    )

    drug_results = []
    for model_id in combined_latent_df["ModelID"].unique():
        df = compute_and_plot_latent_scores(
            model_id, combined_latent_df, drug_max, "name", "pearson_correlation", "Drug",
            make_plot=False,
        )
        drug_results.append(df)
    drug_results_df = pd.concat(drug_results, ignore_index=True) if drug_results else pd.DataFrame()
    print(f"Drug annotations: {drug_results_df.shape}")
else:
    drug_results_df = pd.DataFrame()
    print(
        f"{drug_correlation_path} not found — skipping drug annotation. "
        "Run 4.drug-dependency/scripts/5.drug_latent_dim_correlation.py first to enable it."
    )

Processing pathway: idasanutlin
Assigned latent dimension 4 for model nmf, 25, init 0 to pathway idasanutlin.
Processing pathway: AMG-232
Assigned latent dimension 8 for model ica, 10, init 0 to pathway AMG-232.
Processing pathway: CGM097
Assigned latent dimension 29 for model nmf, 45, init 0 to pathway CGM097.
Processing pathway: XL-647
Assigned latent dimension 9 for model nmf, 50, init 0 to pathway XL-647.
Processing pathway: nutlin-3
Assigned latent dimension 29 for model nmf, 30, init 0 to pathway nutlin-3.
Processing pathway: poziotinib
Assigned latent dimension 10 for model nmf, 18, init 0 to pathway poziotinib.
Processing pathway: atiprimod
Assigned latent dimension 72 for model ica, 100, init 0 to pathway atiprimod.
Processing pathway: ibrutinib
Assigned latent dimension 6 for model nmf, 12, init 0 to pathway ibrutinib.
Processing pathway: AZ-628
Assigned latent dimension 16 for model nmf, 60, init 0 to pathway AZ-628.
Processing pathway: LGX818
Assigned latent dimension 19 fo

Assigned latent dimension 69 for model nmf, 80, init 0 to pathway oligomycin-a.
Processing pathway: prednisolone-acetate
Assigned latent dimension 3 for model nmf, 6, init 0 to pathway prednisolone-acetate.
Processing pathway: alvespimycin
Assigned latent dimension 30 for model ica, 100, init 0 to pathway alvespimycin.
Processing pathway: genipin
Assigned latent dimension 5 for model nmf, 8, init 0 to pathway genipin.
Processing pathway: U-18666A
Assigned latent dimension 35 for model nmf, 80, init 0 to pathway U-18666A.
Processing pathway: saracatinib
Assigned latent dimension 6 for model nmf, 9, init 0 to pathway saracatinib.
Processing pathway: VU0238429
Assigned latent dimension 76 for model nmf, 90, init 0 to pathway VU0238429.
Processing pathway: CH5132799
Assigned latent dimension 10 for model ica, 20, init 0 to pathway CH5132799.
Processing pathway: WZ-4002
Assigned latent dimension 56 for model ica, 90, init 0 to pathway WZ-4002.
Processing pathway: TAK-901
Assigned latent dim

Assigned latent dimension 3 for model ica, 7, init 0 to pathway SB-505124.
Processing pathway: medroxyprogesterone-acetate
Assigned latent dimension 42 for model ica, 200, init 0 to pathway medroxyprogesterone-acetate.
Processing pathway: dienogest
Assigned latent dimension 0 for model nmf, 90, init 0 to pathway dienogest.
Processing pathway: pacritinib
Assigned latent dimension 5 for model pca, 8, init 0 to pathway pacritinib.
Processing pathway: CYC116
Assigned latent dimension 17 for model nmf, 40, init 0 to pathway CYC116.
Processing pathway: PF-573228
Assigned latent dimension 44 for model nmf, 150, init 0 to pathway PF-573228.
Processing pathway: bromosporine
Assigned latent dimension 5 for model nmf, 12, init 0 to pathway bromosporine.
Processing pathway: TAK-285
Assigned latent dimension 3 for model pca, 8, init 0 to pathway TAK-285.
Processing pathway: bitopertin
Assigned latent dimension 10 for model nmf, 35, init 0 to pathway bitopertin.
Processing pathway: ZM-306416
Assigne

Assigned latent dimension 17 for model ica, 90, init 0 to pathway alfacalcidol.
Processing pathway: eltrombopag
Assigned latent dimension 0 for model ica, 100, init 0 to pathway eltrombopag.
Processing pathway: bisindolylmaleimide-ix
Assigned latent dimension 23 for model nmf, 90, init 0 to pathway bisindolylmaleimide-ix.
Processing pathway: ftorafur
Assigned latent dimension 35 for model nmf, 70, init 0 to pathway ftorafur.
Processing pathway: abemaciclib
Assigned latent dimension 4 for model nmf, 16, init 0 to pathway abemaciclib.
Processing pathway: icotinib
Assigned latent dimension 4 for model pca, 7, init 0 to pathway icotinib.
Processing pathway: trimethoquinol
Assigned latent dimension 32 for model nmf, 60, init 0 to pathway trimethoquinol.
Processing pathway: hypoestoxide
Assigned latent dimension 33 for model ica, 45, init 0 to pathway hypoestoxide.
Processing pathway: tipifarnib
Assigned latent dimension 20 for model nmf, 30, init 0 to pathway tipifarnib.
Processing pathway:

Assigned latent dimension 37 for model nmf, 80, init 0 to pathway cyclovalone.
Processing pathway: halcinonide
Assigned latent dimension 47 for model nmf, 200, init 0 to pathway halcinonide.
Processing pathway: dihydroartemisinin
Assigned latent dimension 5 for model nmf, 40, init 0 to pathway dihydroartemisinin.
Processing pathway: bisacodyl
Assigned latent dimension 6 for model nmf, 50, init 0 to pathway bisacodyl.
Processing pathway: SB-203580
Assigned latent dimension 4 for model pca, 14, init 0 to pathway SB-203580.
Processing pathway: fexinidazole
Assigned latent dimension 59 for model nmf, 150, init 0 to pathway fexinidazole.
Processing pathway: NSC-319726
Assigned latent dimension 165 for model pca, 200, init 0 to pathway NSC-319726.
Processing pathway: nemorubicin
Assigned latent dimension 105 for model ica, 150, init 0 to pathway nemorubicin.
Processing pathway: RITA
Assigned latent dimension 84 for model nmf, 150, init 0 to pathway RITA.
Processing pathway: CR8-(R)
Assigned 

Assigned latent dimension 36 for model ica, 80, init 0 to pathway ryuvidine.
Processing pathway: maropitant
Assigned latent dimension 82 for model pca, 150, init 0 to pathway maropitant.
Processing pathway: VE-821
Assigned latent dimension 48 for model nmf, 50, init 0 to pathway VE-821.
Processing pathway: voxtalisib
Assigned latent dimension 10 for model ica, 18, init 0 to pathway voxtalisib.
Processing pathway: acenocoumarol
Assigned latent dimension 0 for model ica, 16, init 0 to pathway acenocoumarol.
Processing pathway: risperidone
Assigned latent dimension 166 for model nmf, 200, init 0 to pathway risperidone.
Processing pathway: GDC-0980
Assigned latent dimension 41 for model ica, 60, init 0 to pathway GDC-0980.
Processing pathway: dydrogesterone
Assigned latent dimension 33 for model ica, 90, init 0 to pathway dydrogesterone.
Processing pathway: gamma-aminobutyric-acid
Assigned latent dimension 85 for model nmf, 90, init 0 to pathway gamma-aminobutyric-acid.
Processing pathway:

... (15 outputs left out)

No unused latent dimensions left for pathway empagliflozin.
Processing pathway: zolmitriptan
No unused latent dimensions left for pathway zolmitriptan.
Processing pathway: SKF-89976A
No unused latent dimensions left for pathway SKF-89976A.
Processing pathway: rosmarinic-acid
No unused latent dimensions left for pathway rosmarinic-acid.
Processing pathway: methacholine
Assigned latent dimension 16 for model ica, 200, init 0 to pathway methacholine.
Processing pathway: flavoxate
No unused latent dimensions left for pathway flavoxate.
Processing pathway: EPZ004777
No unused latent dimensions left for pathway EPZ004777.
Processing pathway: selegiline
Assigned latent dimension 33 for model pca, 200, init 0 to pathway selegiline.
Processing pathway: 3-AQC
No unused latent dimensions left for pathway 3-AQC.
Processing pathway: anisotropine
No unused latent dimensions left for pathway anisotropine.
Processing pathway: tiapride
No unused latent dimensions left for pathway tiapride.
Processing p

No unused latent dimensions left for pathway BMS-649.
Processing pathway: examorelin
No unused latent dimensions left for pathway examorelin.
Processing pathway: carmoxirole
No unused latent dimensions left for pathway carmoxirole.
Processing pathway: GSK650394
No unused latent dimensions left for pathway GSK650394.
Processing pathway: 3-indolebutyric-acid
No unused latent dimensions left for pathway 3-indolebutyric-acid.
Processing pathway: lacidipine
No unused latent dimensions left for pathway lacidipine.
Processing pathway: pregabalin
No unused latent dimensions left for pathway pregabalin.
Processing pathway: benidipine
No unused latent dimensions left for pathway benidipine.
Processing pathway: ferulic-acid
No unused latent dimensions left for pathway ferulic-acid.
Processing pathway: naringin
No unused latent dimensions left for pathway naringin.
Processing pathway: tofisopam
No unused latent dimensions left for pathway tofisopam.
Processing pathway: fluoxetine
No unused latent 

S462
ST88-14
STS-26T
iHSC1L_C3
iHSC1L_F12


iPN1L
iPN2J
iPN3D
Drug annotations: (23808, 10)


### Save top associations per sample

In [6]:
top_pathways_per_sample = (
    pathway_results_df.sort_values(["ModelID", "gene_set_library", "pathway_score"], ascending=[True, True, False])
    .groupby(["ModelID", "gene_set_library"])
    .head(20)
    if not pathway_results_df.empty else pathway_results_df
)
top_pathways_per_sample.to_parquet(data_results_dir / "NF1_data_top_pathways.parquet", index=False)

if not drug_results_df.empty:
    top_drugs_per_sample = (
        drug_results_df.sort_values(["ModelID", "pathway_score"], ascending=[True, False])
        .groupby("ModelID")
        .head(20)
    )
    top_drugs_per_sample.to_parquet(data_results_dir / "NF1_data_top_drugs.parquet", index=False)

print(f"Saved annotated results to {data_results_dir}")

Saved annotated results to data/largaespada/results
